**HARGA KARBON (RUPIAH KONSTAN 2010) + PENERAPAN KE NILAI DEFORESTASI**

Langkah:
1. Susun seri SCC tahunan 2010-2025 (2007 USD) dari IWG (2016) TSD
   Appendix A dengan tiga skenario discount rate untuk sensitivitas.
2. Sesuaikan basis dolar: 2007 USD -> 2010 USD, dengan rasio CPI-U AS
   resmi BLS (2007=207.342, 2010=218.056).
3. Konversi ke Rupiah dengan SATU kurs tetap (kurs 2010 = 8.991, dari
   Kurs.xlsx) (BUKAN kurs tahun berjalan, supaya tidak mencampur
   fluktuasi nilai tukar ke dalam nilai riil).
4. Kalikan ke co2e_deforestasi_Mg dari panel_deforestasi_provinsi_2010_2025.csv

Output:
nilai deforestasi dalam Rupiah konstan 2010, per provinsi-tahun.

In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
FOLDER = "/content/drive/MyDrive/DatabaseSkripsi/Outputs"
FOLDER_DEFORESTASI = "/content/drive/MyDrive/DatabaseSkripsi/Outputs"

Mounted at /content/drive


In [ ]:
# =====================================================================
# 1. SERI SCC TAHUNAN 2010-2025 (2007 USD/ton CO2)
#    Sumber: IWG (2016), Technical Support Document: Technical Update
#    of the Social Cost of Carbon for Regulatory Impact Analysis Under
#    Executive Order 12866, Appendix A, Table A1.
# =====================================================================
scc_2007usd = pd.DataFrame({
    "year": list(range(2010, 2026)),
    "scc_5pct":   [10, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 13, 13, 13, 14],
    "scc_3pct":   [31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 42, 43, 44, 45, 46],
    "scc_2_5pct": [50, 51, 53, 54, 55, 56, 57, 59, 60, 61, 62, 63, 64, 65, 66, 68],
})

In [ ]:
# =====================================================================
# 2. FAKTOR PENYESUAIAN BASIS DOLAR (2007 -> 2010)
#    CPI-U AS resmi BLS: 2007 = 207.342, 2010 = 218.056
# =====================================================================
CPI_2007 = 207.342
CPI_2010 = 218.056
FAKTOR_DEFLATOR = CPI_2010 / CPI_2007
print(f"Faktor deflator AS (2007->2010): {FAKTOR_DEFLATOR:.5f}")

Faktor deflator AS (2007->2010): 1.05167


In [ ]:
# =====================================================================
# 3. KURS TETAP (2010) -- dari Kurs.xlsx
# =====================================================================
KURS_2010 = 8991
print(f"Kurs tetap yang dipakai (2010): Rp {KURS_2010:,}/USD")

Kurs tetap yang dipakai (2010): Rp 8,991/USD


In [ ]:
# =====================================================================
# 4. HITUNG HARGA KARBON DALAM RUPIAH KONSTAN 2010
# =====================================================================
harga_karbon = scc_2007usd.copy()
for kolom in ["scc_5pct", "scc_3pct", "scc_2_5pct"]:
    nama_baru = kolom.replace("scc_", "harga_rp_")
    harga_karbon[nama_baru] = harga_karbon[kolom] * FAKTOR_DEFLATOR * KURS_2010

print("\n=== HARGA KARBON PER TON CO2 (Rupiah konstan 2010) ===")
print(harga_karbon[["year", "harga_rp_5pct", "harga_rp_3pct", "harga_rp_2_5pct"]]
      .to_string(index=False, float_format=lambda x: f"{x:,.0f}"))

harga_karbon.to_csv(f"{FOLDER}/harga_karbon_2010_2025.csv", index=False)
print(f"\nDisimpan ke: {FOLDER}/harga_karbon_2010_2025.csv")


=== HARGA KARBON PER TON CO2 (Rupiah konstan 2010) ===
 year  harga_rp_5pct  harga_rp_3pct  harga_rp_2_5pct
 2010         94,556        293,123          472,780
 2011        104,012        302,579          482,235
 2012        104,012        312,035          501,146
 2013        104,012        321,490          510,602
 2014        104,012        330,946          520,058
 2015        104,012        340,401          529,513
 2016        104,012        359,313          538,969
 2017        104,012        368,768          557,880
 2018        113,467        378,224          567,336
 2019        113,467        387,679          576,791
 2020        113,467        397,135          586,247
 2021        113,467        397,135          595,702
 2022        122,923        406,590          605,158
 2023        122,923        416,046          614,614
 2024        122,923        425,502          624,069
 2025        132,378        434,957          642,980

Disimpan ke: /content/drive/MyDrive/Databa

In [ ]:
# =====================================================================
# 5. PANEL DEFORESTASI
# =====================================================================
deforestasi = pd.read_csv(f"{FOLDER_DEFORESTASI}/DefosetasiProvinsi_Panel.csv")

print(f"\nJumlah baris panel deforestasi sebelum join: {len(deforestasi)}")

panel_gabungan = deforestasi.merge(
    harga_karbon[["year", "harga_rp_5pct", "harga_rp_3pct", "harga_rp_2_5pct"]],
    on="year",
    how="left",
)

# Cek apakah ada tahun yang gagal mendapat harga
tidak_dapat_harga = panel_gabungan[panel_gabungan["harga_rp_3pct"].isnull()]
if len(tidak_dapat_harga) > 0:
    print(f"\nPERINGATAN -- {len(tidak_dapat_harga)} baris tidak mendapat harga karbon:")
    print(tidak_dapat_harga[["provinsi", "year"]].to_string(index=False))
else:
    print("Semua baris berhasil mendapat harga karbon.")

# --- Nilai deforestasi Rupiah, tiga skenario (3% = utama)
panel_gabungan["nilai_deforestasi_rp_3pct"] = (
    panel_gabungan["co2e_deforestasi_Mg"] * panel_gabungan["harga_rp_3pct"]
)
panel_gabungan["nilai_deforestasi_rp_5pct"] = (
    panel_gabungan["co2e_deforestasi_Mg"] * panel_gabungan["harga_rp_5pct"]
)
panel_gabungan["nilai_deforestasi_rp_2_5pct"] = (
    panel_gabungan["co2e_deforestasi_Mg"] * panel_gabungan["harga_rp_2_5pct"]
)




Jumlah baris panel deforestasi sebelum join: 544
Semua baris berhasil mendapat harga karbon.


In [ ]:
# =====================================================================
# 6. VERIFIKASI DAN SIMPAN
# =====================================================================
print(f"\nJumlah baris panel akhir: {len(panel_gabungan)}")
print(f"Jumlah provinsi          : {panel_gabungan['provinsi'].nunique()}")
print(f"Rentang tahun            : {panel_gabungan['year'].min()}-{panel_gabungan['year'].max()}")

print("\nContoh hasil (5 baris pertama):")
kolom_tampil = ["provinsi", "year", "co2e_deforestasi_Mg",
                 "harga_rp_3pct", "nilai_deforestasi_rp_3pct"]
print(panel_gabungan[kolom_tampil].head().to_string(index=False))

output_path = f"{FOLDER_DEFORESTASI}/panel_deforestasi_nilai_rupiah_2010_2025.csv"
panel_gabungan.to_csv(output_path, index=False)
print(f"\nDisimpan ke: {output_path}")

print(
    "\nCATATAN METODOLOGI: nilai_deforestasi_rp_3pct dipakai sebagai skenario "
    "utama (discount rate 3%, konvensi paling umum di literatur SCC). Dua "
    "kolom lain (5% dan 2.5%) disediakan untuk uji sensitivitas di bab "
    "pembahasan -- discount rate lebih tinggi (5%) menghasilkan nilai "
    "kerugian lebih rendah (bobot lebih kecil untuk kerugian masa depan), "
    "dan sebaliknya untuk 2.5%."
)


Jumlah baris panel akhir: 544
Jumlah provinsi          : 34
Rentang tahun            : 2010-2025

Contoh hasil (5 baris pertama):
provinsi  year  co2e_deforestasi_Mg  harga_rp_3pct  nilai_deforestasi_rp_3pct
    Aceh  2010         2.478091e+07  293123.372862               7.263865e+12
    Aceh  2011         3.027368e+07  302578.965535               9.160179e+12
    Aceh  2012         3.082557e+07  312034.558208               9.618643e+12
    Aceh  2013         2.296889e+07  321490.150881               7.384273e+12
    Aceh  2014         3.429021e+07  330945.743554               1.134820e+13

Disimpan ke: /content/drive/MyDrive/DatabaseSkripsi/Outputs/panel_deforestasi_nilai_rupiah_2010_2025.csv

CATATAN METODOLOGI: nilai_deforestasi_rp_3pct dipakai sebagai skenario utama (discount rate 3%, konvensi paling umum di literatur SCC). Dua kolom lain (5% dan 2.5%) disediakan untuk uji sensitivitas di bab pembahasan -- discount rate lebih tinggi (5%) menghasilkan nilai kerugian lebih rendah (